# Polymarket Whale Monitor - Historical Backtest

This notebook allows you to **validate the Fresh Whale detection logic** by analyzing historical trades on Polymarket.

## What This Does

- Fetches historical trades over a configurable time period
- Analyzes each account's state **at the time of the trade** (not current state)
- Identifies "Fresh Whales" - new accounts making large bets
- Generates detailed reports and CSV exports

## Data Sources
- **Activity Subgraph**: Historical trade data (splits)
- **Gamma API**: Market information
- **Data API**: User trading history

## Why Point-in-Time Analysis Matters

If an account had only 2 trades when they made a $50k bet, we show "2 prior trades" - even if that account now has 100+ trades. This validates that real-time monitoring would have caught them.

---

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install -q requests pandas

print("Dependencies installed successfully!")

## Step 2: Configuration

Adjust these settings to customize your backtest.

In [ ]:
# =============================================================================
# BACKTEST CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# How many days to look back
LOOKBACK_DAYS = 7

# Minimum trade value to analyze (in USD)
MIN_TRADE_VALUE_USD = 10000  # $10,000 default

# "Fresh Account" Detection Criteria:
# An account is flagged if EITHER condition is met:
MAX_HISTORICAL_TRADES = 5   # Fewer than this many prior trades
NEW_ACCOUNT_HOURS = 72      # OR account age less than this (hours)

# Output settings
EXPORT_WHALES_CSV = True           # Export detected whales to CSV
WHALES_CSV_FILENAME = "fresh_whales.csv"

EXPORT_ALL_TRADES_CSV = False      # Export ALL large trades (for analysis)
ALL_TRADES_CSV_FILENAME = "all_large_trades.csv"

# API settings (usually don't need to change)
BATCH_SIZE = 100           # Trades per API call
MAX_TRADES = 10000         # Safety limit
RATE_LIMIT_DELAY = 0.3     # Seconds between API calls

print("Configuration loaded!")
print(f"  - Lookback period: {LOOKBACK_DAYS} days")
print(f"  - Minimum trade value: ${MIN_TRADE_VALUE_USD:,}")
print(f"  - Fresh account: <{MAX_HISTORICAL_TRADES} trades OR <{NEW_ACCOUNT_HOURS}h old")

## Step 3: Load Backtest Engine

Run this cell to load all the analysis code.

In [ ]:
import os
import sys
import time
import json
import csv
from datetime import datetime, timedelta, timezone
from typing import Optional, Dict, List, Any, Set, Tuple
from dataclasses import dataclass, field
from collections import defaultdict
from IPython.display import display, HTML, clear_output

import requests
import pandas as pd

# =============================================================================
# CONFIGURATION CLASS
# =============================================================================

@dataclass
class BacktestConfig:
    """Configuration for historical backtest."""
    # API Endpoints
    ACTIVITY_SUBGRAPH_URL: str = (
        "https://api.goldsky.com/api/public/"
        "project_cl6mb8i9h0003e201j6li0diw/subgraphs/activity-subgraph/0.0.4/gn"
    )
    GAMMA_API_URL: str = "https://gamma-api.polymarket.com"
    DATA_API_URL: str = "https://data-api.polymarket.com"
    
    MIN_TRADE_VALUE_USD: float = 10_000.0
    MAX_HISTORICAL_TRADES: int = 5
    NEW_ACCOUNT_HOURS: int = 72
    LOOKBACK_DAYS: int = 7
    BATCH_SIZE: int = 100
    MAX_TRADES_TO_ANALYZE: int = 10000
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 3
    RETRY_DELAY: int = 5
    RATE_LIMIT_DELAY: float = 0.3

# =============================================================================
# DATA MODELS
# =============================================================================

@dataclass
class HistoricalTrade:
    """A trade with point-in-time account analysis."""
    id: str
    user_address: str
    market_id: str
    market_title: str
    outcome: str
    amount: float
    price: float
    value_usd: float
    timestamp: int
    tx_hash: str
    account_trades_before: int
    account_first_trade_ts: Optional[int]
    account_age_hours_at_trade: Optional[float]
    is_fresh_whale: bool
    detection_reason: str

    @property
    def formatted_time(self) -> str:
        return datetime.fromtimestamp(
            self.timestamp, tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")

    @property
    def polymarket_profile_url(self) -> str:
        return f"https://polymarket.com/profile/{self.user_address}"

    def to_dict(self) -> Dict[str, Any]:
        return {
            "timestamp": self.formatted_time,
            "unix_timestamp": self.timestamp,
            "value_usd": self.value_usd,
            "market_title": self.market_title,
            "outcome": self.outcome,
            "price": self.price,
            "shares": self.amount,
            "user_address": self.user_address,
            "account_trades_before": self.account_trades_before,
            "account_age_hours": self.account_age_hours_at_trade,
            "is_fresh_whale": self.is_fresh_whale,
            "detection_reason": self.detection_reason,
            "tx_hash": self.tx_hash,
            "profile_url": self.polymarket_profile_url
        }


@dataclass
class BacktestResults:
    """Summary of backtest results."""
    start_time: datetime
    end_time: datetime
    config: BacktestConfig
    total_trades_fetched: int = 0
    large_trades_analyzed: int = 0
    fresh_whales_detected: int = 0
    fresh_whale_trades: List[HistoricalTrade] = field(default_factory=list)
    all_large_trades: List[HistoricalTrade] = field(default_factory=list)
    unique_whale_addresses: Set[str] = field(default_factory=set)
    total_whale_volume_usd: float = 0.0
    trades_by_day: Dict[str, int] = field(default_factory=dict)
    whales_by_market: Dict[str, int] = field(default_factory=dict)

    def add_whale(self, trade: HistoricalTrade):
        self.fresh_whale_trades.append(trade)
        self.fresh_whales_detected += 1
        self.unique_whale_addresses.add(trade.user_address)
        self.total_whale_volume_usd += trade.value_usd
        day_key = datetime.fromtimestamp(trade.timestamp, tz=timezone.utc).strftime("%Y-%m-%d")
        self.trades_by_day[day_key] = self.trades_by_day.get(day_key, 0) + 1
        market_key = trade.market_title[:50]
        self.whales_by_market[market_key] = self.whales_by_market.get(market_key, 0) + 1

    def to_dataframe(self) -> pd.DataFrame:
        if not self.fresh_whale_trades:
            return pd.DataFrame()
        return pd.DataFrame([t.to_dict() for t in self.fresh_whale_trades])

    def all_trades_to_dataframe(self) -> pd.DataFrame:
        if not self.all_large_trades:
            return pd.DataFrame()
        return pd.DataFrame([t.to_dict() for t in self.all_large_trades])


# =============================================================================
# API CLIENT
# =============================================================================

class HistoricalClient:
    """Client for fetching historical data from Polymarket APIs."""

    def __init__(self, config: BacktestConfig):
        self.config = config
        self.session = requests.Session()
        self._market_cache: Dict[str, Dict] = {}
        self._user_activity_cache: Dict[str, List[Dict]] = {}

    def _request_with_retry(self, method: str, url: str, **kwargs) -> Optional[requests.Response]:
        kwargs.setdefault("timeout", self.config.REQUEST_TIMEOUT)
        for attempt in range(self.config.MAX_RETRIES):
            try:
                response = self.session.request(method, url, **kwargs)
                response.raise_for_status()
                return response
            except requests.exceptions.RequestException as e:
                print(f"Request failed (attempt {attempt + 1}): {e}")
                if attempt < self.config.MAX_RETRIES - 1:
                    time.sleep(self.config.RETRY_DELAY)
        return None

    def _graphql_query(self, query: str, variables: Optional[Dict] = None) -> Optional[Dict]:
        payload = {"query": query}
        if variables:
            payload["variables"] = variables

        response = self._request_with_retry(
            "POST",
            self.config.ACTIVITY_SUBGRAPH_URL,
            json=payload,
            headers={"Content-Type": "application/json"}
        )

        if not response:
            return None

        result = response.json()
        if "errors" in result:
            print(f"GraphQL errors: {result['errors']}")
            return None

        return result.get("data")

    def get_historical_splits(self, start_timestamp: int, end_timestamp: int, min_amount_raw: int, progress_callback=None) -> List[Dict]:
        """Fetch historical splits (trades) from Activity Subgraph with pagination."""
        all_splits = []

        query_initial = """
        query GetHistoricalSplits($start: BigInt!, $end: BigInt!, $minAmount: BigInt!, $first: Int!) {
            splits(
                first: $first,
                orderBy: timestamp,
                orderDirection: desc,
                where: {
                    timestamp_gte: $start,
                    timestamp_lte: $end,
                    amount_gte: $minAmount
                }
            ) {
                id
                timestamp
                stakeholder
                condition
                amount
            }
        }
        """

        query_paginated = """
        query GetHistoricalSplits($start: BigInt!, $end: BigInt!, $minAmount: BigInt!, $first: Int!, $lastId: String!) {
            splits(
                first: $first,
                orderBy: timestamp,
                orderDirection: desc,
                where: {
                    timestamp_gte: $start,
                    timestamp_lte: $end,
                    amount_gte: $minAmount,
                    id_lt: $lastId
                }
            ) {
                id
                timestamp
                stakeholder
                condition
                amount
            }
        }
        """

        variables = {
            "start": str(start_timestamp),
            "end": str(end_timestamp),
            "minAmount": str(min_amount_raw),
            "first": self.config.BATCH_SIZE
        }

        data = self._graphql_query(query_initial, variables)

        if not data or "splits" not in data:
            return []

        splits = data["splits"]
        all_splits.extend(splits)

        if progress_callback:
            progress_callback(len(all_splits))

        while len(splits) == self.config.BATCH_SIZE:
            if len(all_splits) >= self.config.MAX_TRADES_TO_ANALYZE:
                print(f"Reached max trades limit ({self.config.MAX_TRADES_TO_ANALYZE})")
                break

            last_id = splits[-1]["id"]
            variables["lastId"] = last_id

            time.sleep(self.config.RATE_LIMIT_DELAY)
            data = self._graphql_query(query_paginated, variables)

            if not data or "splits" not in data:
                break

            splits = data["splits"]
            all_splits.extend(splits)

            if progress_callback:
                progress_callback(len(all_splits))

        return all_splits

    def get_market_info(self, condition_id: str) -> Optional[Dict]:
        """Get market information from Gamma API by condition ID."""
        if condition_id in self._market_cache:
            return self._market_cache[condition_id]

        time.sleep(self.config.RATE_LIMIT_DELAY)

        url = f"{self.config.GAMMA_API_URL}/markets"
        params = {"conditionId": condition_id}

        response = self._request_with_retry("GET", url, params=params)

        if response and response.status_code == 200:
            markets = response.json()
            if markets and len(markets) > 0:
                market = markets[0]
                self._market_cache[condition_id] = market
                return market

        return None

    def get_user_activity(self, user_address: str) -> List[Dict]:
        """Get user's trading activity from Data API."""
        address = user_address.lower()

        if address in self._user_activity_cache:
            return self._user_activity_cache[address]

        time.sleep(self.config.RATE_LIMIT_DELAY)

        url = f"{self.config.DATA_API_URL}/activity"
        params = {"user": address, "limit": 1000}

        response = self._request_with_retry("GET", url, params=params)

        if response and response.status_code == 200:
            activities = response.json()
            self._user_activity_cache[address] = activities
            return activities

        return []

    def get_account_trades_before(self, address: str, before_timestamp: int) -> Tuple[int, Optional[int]]:
        """Get the number of trades an account had BEFORE a specific timestamp."""
        activities = self.get_user_activity(address)

        if not activities:
            return 0, None

        trades = [a for a in activities if a.get("type") == "TRADE"]

        if not trades:
            return 0, None

        trades_sorted = sorted(trades, key=lambda x: x.get("timestamp", 0))
        first_ts = trades_sorted[0].get("timestamp") if trades_sorted else None
        trades_before = [t for t in trades_sorted if t.get("timestamp", 0) < before_timestamp]

        return len(trades_before), first_ts


# =============================================================================
# BACKTEST ENGINE
# =============================================================================

class BacktestEngine:
    """Engine for running historical backtest analysis."""

    def __init__(self, config: BacktestConfig):
        self.config = config
        self.client = HistoricalClient(config)

    def run(self, start_date: Optional[datetime] = None, end_date: Optional[datetime] = None, show_progress: bool = True) -> BacktestResults:
        if end_date is None:
            end_date = datetime.now(timezone.utc)
        if start_date is None:
            start_date = end_date - timedelta(days=self.config.LOOKBACK_DAYS)

        start_ts = int(start_date.timestamp())
        end_ts = int(end_date.timestamp())

        results = BacktestResults(
            start_time=start_date,
            end_time=end_date,
            config=self.config
        )

        if show_progress:
            print("="*60)
            print("Starting Historical Backtest")
            print(f"Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
            print(f"Minimum trade value: ${self.config.MIN_TRADE_VALUE_USD:,.0f}")
            print("="*60)
            print("\nFetching historical trades...")

        min_amount_raw = int(self.config.MIN_TRADE_VALUE_USD * 1_000_000)

        def progress_callback(count):
            if show_progress:
                print(f"  Fetched {count} splits...", end="\r")

        splits = self.client.get_historical_splits(
            start_timestamp=start_ts,
            end_timestamp=end_ts,
            min_amount_raw=min_amount_raw,
            progress_callback=progress_callback
        )

        results.total_trades_fetched = len(splits)

        if show_progress:
            print(f"\nFound {len(splits)} trades above ${self.config.MIN_TRADE_VALUE_USD:,.0f}")
            print("\nAnalyzing trades for Fresh Whale patterns...")

        for i, split in enumerate(splits):
            if show_progress and (i + 1) % 25 == 0:
                print(f"  Processed {i + 1}/{len(splits)} trades...", end="\r")

            historical_trade = self._analyze_split(split)
            if historical_trade:
                results.all_large_trades.append(historical_trade)
                results.large_trades_analyzed += 1

                if historical_trade.is_fresh_whale:
                    results.add_whale(historical_trade)

        if show_progress:
            print(f"\n\nAnalysis complete! Found {results.fresh_whales_detected} Fresh Whales.")

        return results

    def _analyze_split(self, split: Dict) -> Optional[HistoricalTrade]:
        """Analyze a single split for Fresh Whale status."""
        try:
            split_id = split["id"]
            timestamp = int(split["timestamp"])
            stakeholder = split["stakeholder"]
            condition = split["condition"]
            amount_raw = int(split["amount"])
            value_usd = amount_raw / 1_000_000

            market_info = self.client.get_market_info(condition)
            if market_info:
                market_title = market_info.get("question", "Unknown Market")
                market_id = market_info.get("id", condition)
            else:
                market_title = f"Market {condition[:16]}..."
                market_id = condition

            trades_before, first_trade_ts = self.client.get_account_trades_before(
                stakeholder, timestamp
            )

            if first_trade_ts:
                age_seconds = timestamp - first_trade_ts
                age_hours = max(0, age_seconds / 3600)
            else:
                age_hours = None

            is_fresh = False
            reason = ""

            if trades_before < self.config.MAX_HISTORICAL_TRADES:
                is_fresh = True
                reason = f"Only {trades_before} prior trades"
            elif age_hours is not None and age_hours < self.config.NEW_ACCOUNT_HOURS:
                is_fresh = True
                reason = f"Account only {age_hours:.1f} hours old at time of trade"

            return HistoricalTrade(
                id=split_id,
                user_address=stakeholder,
                market_id=market_id,
                market_title=market_title,
                outcome="Position",
                amount=value_usd,
                price=1.0,
                value_usd=value_usd,
                timestamp=timestamp,
                tx_hash=split_id.split("_")[0] if "_" in split_id else split_id,
                account_trades_before=trades_before,
                account_first_trade_ts=first_trade_ts,
                account_age_hours_at_trade=age_hours,
                is_fresh_whale=is_fresh,
                detection_reason=reason
            )

        except (KeyError, ValueError, TypeError) as e:
            print(f"Failed to analyze split: {e}")
            return None


def display_results_summary(results: BacktestResults):
    """Display results in a nice HTML format for notebooks."""
    avg_trade = results.total_whale_volume_usd / results.fresh_whales_detected if results.fresh_whales_detected > 0 else 0

    html = f"""
    <div style="font-family: monospace; background: #1a1a2e; color: #eee; padding: 20px; border-radius: 10px; margin: 10px 0;">
        <h2 style="color: #00d4ff; border-bottom: 2px solid #00d4ff; padding-bottom: 10px;">Backtest Results Summary</h2>

        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin: 20px 0;">
            <div style="background: #16213e; padding: 15px; border-radius: 8px;">
                <h3 style="color: #00d4ff; margin-top: 0;">Period</h3>
                <p>{results.start_time.strftime('%Y-%m-%d')} to {results.end_time.strftime('%Y-%m-%d')}</p>
                <p>Min Trade: <strong>${results.config.MIN_TRADE_VALUE_USD:,.0f}</strong></p>
                <p>Fresh = &lt;{results.config.MAX_HISTORICAL_TRADES} trades OR &lt;{results.config.NEW_ACCOUNT_HOURS}h old</p>
            </div>

            <div style="background: #16213e; padding: 15px; border-radius: 8px;">
                <h3 style="color: #00d4ff; margin-top: 0;">Statistics</h3>
                <p>Trades Fetched: <strong>{results.total_trades_fetched:,}</strong></p>
                <p>Large Trades Analyzed: <strong>{results.large_trades_analyzed:,}</strong></p>
                <p style="color: #ff6b6b;">Fresh Whales Detected: <strong>{results.fresh_whales_detected:,}</strong></p>
            </div>
        </div>

        <div style="background: #16213e; padding: 15px; border-radius: 8px; margin: 20px 0;">
            <h3 style="color: #ff6b6b; margin-top: 0;">Whale Summary</h3>
            <p>Unique Addresses: <strong>{len(results.unique_whale_addresses):,}</strong></p>
            <p>Total Volume: <strong style="color: #4ecdc4;">${results.total_whale_volume_usd:,.2f}</strong></p>
            <p>Average Trade Size: <strong>${avg_trade:,.2f}</strong></p>
        </div>
    """

    if results.fresh_whale_trades:
        html += """
        <h3 style="color: #ff6b6b;">Top 10 Largest Fresh Whale Trades</h3>
        <div style="background: #16213e; padding: 15px; border-radius: 8px;">
        """

        sorted_trades = sorted(results.fresh_whale_trades, key=lambda t: t.value_usd, reverse=True)[:10]

        for i, trade in enumerate(sorted_trades, 1):
            html += f"""
            <div style="border-left: 4px solid #ff6b6b; padding-left: 15px; margin: 15px 0;">
                <p style="margin: 0;"><strong style="color: #4ecdc4;">#{i}. ${trade.value_usd:,.2f}</strong> - {trade.formatted_time}</p>
                <p style="margin: 5px 0; color: #aaa;">{trade.market_title[:70]}...</p>
                <p style="margin: 5px 0;">Position: <strong>{trade.outcome}</strong> @ ${trade.price:.3f}</p>
                <p style="margin: 5px 0; color: #ff6b6b;">Reason: {trade.detection_reason}</p>
                <p style="margin: 5px 0; font-size: 0.9em;">Wallet: <code>{trade.user_address[:12]}...{trade.user_address[-8:]}</code></p>
            </div>
            """

        html += "</div>"

    if results.trades_by_day:
        html += """
        <h3 style="color: #00d4ff;">Fresh Whales by Day</h3>
        <div style="background: #16213e; padding: 15px; border-radius: 8px;">
        """
        max_count = max(results.trades_by_day.values()) if results.trades_by_day else 1
        for day in sorted(results.trades_by_day.keys()):
            count = results.trades_by_day[day]
            bar_width = int((count / max_count) * 200) if max_count > 0 else 0
            html += f"""
            <div style="margin: 5px 0; display: flex; align-items: center;">
                <span style="width: 100px;">{day}</span>
                <div style="background: #ff6b6b; height: 20px; width: {bar_width}px; border-radius: 3px;"></div>
                <span style="margin-left: 10px;">{count}</span>
            </div>
            """
        html += "</div>"

    html += "</div>"
    display(HTML(html))


print("Backtest engine loaded successfully!")

## Step 4: Run the Backtest

Execute this cell to run the historical analysis.

In [ ]:
# =============================================================================
# RUN THE BACKTEST
# =============================================================================

# Build configuration from settings
config = BacktestConfig(
    LOOKBACK_DAYS=LOOKBACK_DAYS,
    MIN_TRADE_VALUE_USD=MIN_TRADE_VALUE_USD,
    MAX_HISTORICAL_TRADES=MAX_HISTORICAL_TRADES,
    NEW_ACCOUNT_HOURS=NEW_ACCOUNT_HOURS,
    BATCH_SIZE=BATCH_SIZE,
    MAX_TRADES_TO_ANALYZE=MAX_TRADES,
    RATE_LIMIT_DELAY=RATE_LIMIT_DELAY
)

# Run the backtest
engine = BacktestEngine(config)
results = engine.run(show_progress=True)

# Display formatted results
display_results_summary(results)

## Step 5: View Results as DataFrame

Explore the detected Fresh Whales in a pandas DataFrame.

In [ ]:
# View Fresh Whales as a DataFrame
if results.fresh_whale_trades:
    df_whales = results.to_dataframe()

    # Sort by value descending
    df_whales = df_whales.sort_values('value_usd', ascending=False)

    # Display with nice formatting
    print(f"Found {len(df_whales)} Fresh Whale trades:")
    display(df_whales[
        ['timestamp', 'value_usd', 'market_title', 'outcome', 'price',
         'account_trades_before', 'account_age_hours', 'detection_reason']
    ].head(20))
else:
    print("No Fresh Whales detected in this period.")

In [ ]:
# View ALL large trades (including non-whales) for comparison
if results.all_large_trades:
    df_all = results.all_trades_to_dataframe()
    df_all = df_all.sort_values('value_usd', ascending=False)

    print(f"Total large trades analyzed: {len(df_all)}")
    print(f"Fresh Whales: {len(df_all[df_all['is_fresh_whale'] == True])}")
    print(f"Established traders: {len(df_all[df_all['is_fresh_whale'] == False])}")

    # Show comparison
    display(df_all[
        ['timestamp', 'value_usd', 'is_fresh_whale', 'account_trades_before',
         'account_age_hours', 'outcome', 'market_title']
    ].head(20))

## Step 6: Export to CSV

Save results to CSV files for further analysis.

In [ ]:
# Export Fresh Whales to CSV
if EXPORT_WHALES_CSV and results.fresh_whale_trades:
    df_whales = results.to_dataframe()
    df_whales = df_whales.sort_values('value_usd', ascending=False)
    df_whales.to_csv(WHALES_CSV_FILENAME, index=False)
    print(f"Exported {len(df_whales)} Fresh Whale trades to {WHALES_CSV_FILENAME}")

    # Download link for Colab
    try:
        from google.colab import files
        files.download(WHALES_CSV_FILENAME)
    except:
        print(f"File saved locally: {WHALES_CSV_FILENAME}")
else:
    if not results.fresh_whale_trades:
        print("No Fresh Whales to export.")
    else:
        print("Whale CSV export disabled. Set EXPORT_WHALES_CSV = True to enable.")

In [ ]:
# Export ALL large trades to CSV (for analysis)
if EXPORT_ALL_TRADES_CSV and results.all_large_trades:
    df_all = results.all_trades_to_dataframe()
    df_all = df_all.sort_values('value_usd', ascending=False)
    df_all.to_csv(ALL_TRADES_CSV_FILENAME, index=False)
    print(f"Exported {len(df_all)} trades to {ALL_TRADES_CSV_FILENAME}")

    # Download link for Colab
    try:
        from google.colab import files
        files.download(ALL_TRADES_CSV_FILENAME)
    except:
        print(f"File saved locally: {ALL_TRADES_CSV_FILENAME}")
else:
    print("All-trades CSV export disabled. Set EXPORT_ALL_TRADES_CSV = True to enable.")

## Step 7: Custom Analysis (Optional)

Use these cells for custom queries on the data.

In [ ]:
# Example: Find whales who bet on specific topics
if results.fresh_whale_trades:
    df = results.to_dataframe()

    # Filter by keyword in market title
    keyword = "Bitcoin"  # Change this to search for different topics
    matches = df[df['market_title'].str.contains(keyword, case=False, na=False)]

    if len(matches) > 0:
        print(f"Found {len(matches)} Fresh Whale trades related to '{keyword}':")
        display(matches[['timestamp', 'value_usd', 'outcome', 'market_title', 'detection_reason']])
    else:
        print(f"No Fresh Whale trades found related to '{keyword}'")

In [ ]:
# Example: Analyze whale behavior by outcome (Yes vs No)
if results.fresh_whale_trades:
    df = results.to_dataframe()

    # Group by outcome
    outcome_stats = df.groupby('outcome').agg({
        'value_usd': ['count', 'sum', 'mean']
    }).round(2)

    print("Fresh Whale betting patterns:")
    print(f"\nYes bets: {len(df[df['outcome'] == 'Yes'])} trades, ${df[df['outcome'] == 'Yes']['value_usd'].sum():,.2f} total")
    print(f"No bets: {len(df[df['outcome'] == 'No'])} trades, ${df[df['outcome'] == 'No']['value_usd'].sum():,.2f} total")

In [ ]:
# Example: Find repeat whale addresses
if results.fresh_whale_trades:
    df = results.to_dataframe()

    # Count trades per address
    address_counts = df.groupby('user_address').agg({
        'value_usd': ['count', 'sum']
    }).reset_index()
    address_counts.columns = ['address', 'trade_count', 'total_volume']
    address_counts = address_counts.sort_values('total_volume', ascending=False)

    repeat_whales = address_counts[address_counts['trade_count'] > 1]

    if len(repeat_whales) > 0:
        print(f"Found {len(repeat_whales)} addresses with multiple Fresh Whale trades:")
        display(repeat_whales.head(10))
    else:
        print("All Fresh Whales made only one large trade each.")

---

## Understanding the Results

### Key Columns Explained

| Column | Meaning |
|--------|--------|
| `account_trades_before` | Number of trades this wallet had BEFORE this specific trade |
| `account_age_hours` | How old the account was (in hours) when this trade was made |
| `is_fresh_whale` | True if the account was "fresh" at the time of the trade |
| `detection_reason` | Why the account was flagged as a Fresh Whale |

### Validation Logic

An account is flagged as a **Fresh Whale** if, at the time of the trade:
1. They had fewer than `MAX_HISTORICAL_TRADES` prior trades, **OR**
2. Their account was less than `NEW_ACCOUNT_HOURS` hours old

This point-in-time analysis ensures the backtest accurately represents what the real-time monitor would have detected.

---